# 🛗 Elevador Inteligente — Validadores, Máquina de Estados e Scanner

**Disciplina:** Compiladores — Ciência da Computação, 8º semestre
**Atividade:** Aula CP 08 (Prática 3) — Autômatos Finitos na Prática
**Domínio escolhido:** Elevador inteligente

**Integrantes**

| RA | Nome |
|---|---|
| 2392072 | Diego Azevedo Dias Inacio |
| 2334425 | Gabriella Pereira Rodrigues |
| 2367880 | Giovanna Bispo Da Silva |
| 2499948 | José Henrique Guimarães Galvão Silva |

**Como rodar:** menu *Ambiente de execução → Executar tudo*. Só usamos a biblioteca padrão do Python, `ipywidgets` (já vem no Colab) e, opcionalmente, `graphviz` (também já vem no Colab).

**Mapa do notebook**

| Seção | Conteúdo | Requisito |
|---|---|---|
| 0 | Ambiente e classe `AFD` | — |
| 1 | Validador de **andar** | R1, R2, R3 |
| 2 | Validador de **código de manutenção** | R1, R2, R3 |
| 3 | **Máquina de estados** do elevador | R4 |
| 4 | **Mini-scanner** da linguagem de comandos | R5 |
| 5 | **Interface** (ipywidgets) | R6 |
| 6 | **Bônus:** AFD mínimo, desenho com Graphviz, tudo conectado | Bônus |

## ⚙️ Seção 0 — Ambiente e a classe `AFD`

In [ ]:
# ⚙️ CÉLULA 0 — Preparação do ambiente
# Só biblioteca padrão. (ipywidgets e graphviz já vêm instalados no Google Colab.)
import re

print("✅ Ambiente pronto! Vamos construir o Elevador Inteligente.")

In [ ]:
# 🧰 CÉLULA 1 — A classe AFD (o "motor" reutilizável da aula)
# Um AFD é a 5-upla  M = (Q, Σ, δ, q0, F).

class AFD:
    def __init__(self, nome, delta, inicial, finais, classificar):
        self.nome = nome
        self.delta = delta                  # δ: {(estado, classe): próximo}
        self.inicial = inicial              # q0
        self.finais = set(finais)           # F
        self.classificar = classificar      # função: caractere -> classe (Σ)
        self.estados = sorted({e for e, _ in delta} | set(delta.values()))           # Q
        self.alfabeto = sorted({s for _, s in delta})                               # Σ

    def rodar(self, palavra, mostrar=True):
        estado = self.inicial
        if mostrar:
            print(f"🔎 {self.nome} lendo {palavra!r}")
            print(f"   {'passo':5} {'lido':5} {'classe':7} {'de':6} {'para':6}")
        for passo, c in enumerate(palavra, start=1):
            classe = self.classificar(c)
            proximo = self.delta.get((estado, classe))
            if mostrar:
                print(f"   {passo:<5} {c!r:5} {classe:7} {estado:6} {proximo or '💀 morto'}")
            if proximo is None:
                if mostrar:
                    print(f"   ❌ REJEITADA: não há transição de {estado} com {classe!r}\n")
                return False
            estado = proximo
        aceita = estado in self.finais
        if mostrar:
            if aceita:
                print(f"   ✅ ACEITA: terminou em {estado}, que é de aceitação\n")
            else:
                print(f"   ❌ REJEITADA: terminou em {estado}, que NÃO é de aceitação\n")
        return aceita

    def tabela(self):
        """Imprime a tabela de transição: linhas = estados, colunas = classes."""
        lq = max([len(q) for q in self.estados] + [6]) + 1                       # largura da coluna de estados
        lc = max([len(s) for s in self.alfabeto] + [len(v) for v in self.delta.values()] + [4]) + 1
        print(f"📋 Tabela de transição — {self.nome}")
        print("   " + f"{'estado':{lq}}" + "".join(f"{s:>{lc}}" for s in self.alfabeto))
        for q in self.estados:
            marca = ("→" if q == self.inicial else " ") + ("*" if q in self.finais else " ")
            linha = "".join(f"{self.delta.get((q, s), '—'):>{lc}}" for s in self.alfabeto)
            print(f"{marca} {q:{lq}}{linha}")
        print("   (→ = inicial, * = aceitação, — = vai para o estado morto)\n")


def bateria(afd, regex, aceitas, rejeitadas):
    """Bateria de testes AFD × RegEx.
    Confere (1) AFD == RegEx e (2) o resultado esperado de cada palavra.
    Devolve o número de divergências."""
    divergencias = 0
    print(f"🧪 {afd.nome}  |  RegEx: {regex}")
    print(f"   {'palavra':12} {'esperado':9} {'AFD':6} {'RegEx':6}")
    for palavra, esperado in [(p, True) for p in aceitas] + [(p, False) for p in rejeitadas]:
        r_afd = afd.rodar(palavra, mostrar=False)
        r_re = re.fullmatch(regex, palavra) is not None
        ok = (r_afd == r_re == esperado)
        divergencias += (not ok)
        print(f"   {palavra!r:12} {str(esperado):9} {str(r_afd):6} {str(r_re):6} {'✅' if ok else '⚠️ DIVERGE'}")
    print(f"   → {len(aceitas)} aceitas, {len(rejeitadas)} rejeitadas, divergências: {divergencias}\n")
    return divergencias

## 🔢 Seção 1 — Validador de **andar** (R1, R2, R3)

Um andar válido é: `T` (térreo), `S1` ou `S2` (subsolos) ou um número de `1` a `20`.

Os dígitos precisam de **classes finas**, como no exemplo do horário:

| Classe | Caracteres | Por quê |
|---|---|---|
| `0` | 0 | só aparece depois de `1` (10) ou de `2` (20) |
| `1` | 1 | pode ser andar sozinho, começar `10`–`19`, ou ser `S1` |
| `2` | 2 | pode ser andar sozinho, ser `20`, ou ser `S2` |
| `3-9` | 3 … 9 | só andar sozinho |
| `T` | T | térreo |
| `S` | S | subsolo |
| `outro` | qualquer outro | vai para o estado morto |

**Desenho**

```
 ──► ( A0 ) ──T───► (( AT ))
        │
        ├──S───► ( AS ) ──1, 2────────────► (( AF ))
        │
        ├──1───► (( A1 )) ──0,1,2,3-9─────► (( AF ))     # 10 a 19
        │
        ├──2───► (( A2 )) ──0─────────────► (( AF ))     # 20
        │
        └──3-9─► (( AD ))
```

Os estados `AT`, `AD` e `AF` se comportam igual (aceitam e não têm saída). Isso aparece como **AFD mínimo** na Seção 6.

`1` e `2` sozinhos são andares válidos; por isso `A1` e `A2` também são de aceitação.

In [ ]:
# 🔢 CÉLULA 2 — AFD do andar:  T | S1 | S2 | 1..20

def classe_andar(c):
    if c == "0":
        return "0"
    if c == "1":
        return "1"
    if c == "2":
        return "2"
    if c in ("3", "4", "5", "6", "7", "8", "9"):
        return "3-9"
    if c == "T":
        return "T"
    if c == "S":
        return "S"
    return "outro"

delta_andar = {
    ("A0", "T"): "AT",
    ("A0", "S"): "AS",
    ("A0", "1"): "A1",
    ("A0", "2"): "A2",
    ("A0", "3-9"): "AD",
    ("AS", "1"): "AF",                 # S1
    ("AS", "2"): "AF",                 # S2
    ("A2", "0"): "AF",                 # 20
}
for d in ["0", "1", "2", "3-9"]:
    delta_andar[("A1", d)] = "AF"      # 10 a 19

andar = AFD("Andar do elevador", delta_andar, "A0", {"AT", "A1", "A2", "AD", "AF"}, classe_andar)
REGEX_ANDAR = r"T|S[12]|20|1[0-9]|[1-9]"

andar.tabela()

In [ ]:
# 🔎 CÉLULA 3 — Rastreamento passo a passo do andar (R2)
andar.rodar("S2")      # ACEITA: subsolo 2
andar.rodar("21")      # REJEITADA: morre no meio (em A2 não existe seta com '1')
andar.rodar("S")       # REJEITADA: termina em AS, que NÃO é de aceitação

In [ ]:
# 🧪 CÉLULA 4 — Bateria AFD × RegEx do andar (R3)
ACEITAS_ANDAR   = ["T", "S1", "S2", "1", "7", "9", "10", "19", "20"]
REJEITADAS_ANDAR = ["", "0", "21", "S", "S0", "S3", "T1", "01", "100", "t", "s1", "1 ", "30"]

div_andar = bateria(andar, REGEX_ANDAR, ACEITAS_ANDAR, REJEITADAS_ANDAR)

**Por que foram rejeitadas?**

| Palavra | Motivo |
|---|---|
| `21` | em `A2` (leu `2`) só existe seta com `0`; chega `1` e o autômato morre |
| `S` | não morre, mas termina em `AS`, que não é de aceitação (faltou o número) |
| `` (vazia) | termina em `A0`, que não é de aceitação |
| `0` | `A0` não tem seta com `0`; não existe andar 0 (o térreo é `T`) |
| `t`, `s1` | minúsculas caem na classe `outro` |

## 🛠️ Seção 2 — Validador de **código de manutenção** (R1, R2, R3)

Formato: `MNT-` seguido de **exatamente 4 dígitos**, como `MNT-0042`.

| Classe | Caracteres |
|---|---|
| `M`, `N`, `T` | as letras do prefixo (cada uma é uma classe) |
| `-` | hífen |
| `D` | dígito 0–9 |
| `outro` | qualquer outro |

**Desenho**

```
 ──► (M0) ─M─► (M1) ─N─► (M2) ─T─► (M3) ─-─► (M4) ─D─► (M5) ─D─► (M6) ─D─► (M7) ─D─► ((M8))
```

In [ ]:
# 🛠️ CÉLULA 5 — AFD do código de manutenção:  MNT-dddd

def classe_manut(c):
    if c in ("M", "N", "T", "-"):
        return c
    if "0" <= c <= "9":
        return "D"
    return "outro"

delta_manut = {
    ("M0", "M"): "M1",
    ("M1", "N"): "M2",
    ("M2", "T"): "M3",
    ("M3", "-"): "M4",
    ("M4", "D"): "M5",
    ("M5", "D"): "M6",
    ("M6", "D"): "M7",
    ("M7", "D"): "M8",
}
manut = AFD("Código de manutenção", delta_manut, "M0", {"M8"}, classe_manut)
REGEX_MANUT = r"MNT-[0-9]{4}"

manut.tabela()
manut.rodar("MNT-0042")     # ACEITA
manut.rodar("MNT-42")       # REJEITADA: termina em M6 (faltaram 2 dígitos)
manut.rodar("MNT-00425")    # REJEITADA: morre no 5º dígito (em M8 não há seta)

In [ ]:
# 🧪 CÉLULA 6 — Bateria AFD × RegEx do código de manutenção (R3)
ACEITAS_MANUT    = ["MNT-0042", "MNT-0000", "MNT-0001", "MNT-1234", "MNT-9999"]
REJEITADAS_MANUT = ["", "MNT-", "MNT-42", "MNT0042", "mnt-0042", "MNT-00425", "MNT-004A", "ABC-0042"]

div_manut = bateria(manut, REGEX_MANUT, ACEITAS_MANUT, REJEITADAS_MANUT)

## ⚙️ Seção 3 — Máquina de estados do elevador (R4)

Aqui os símbolos não são letras: são **eventos**.

**Regra de segurança do grupo:** o elevador só se move depois de um ciclo completo de porta
(`PARADO → PORTA_ABRINDO → PORTA_ABERTA → PORTA_FECHANDO`). A partir de `PORTA_FECHANDO` ele segue para `SUBINDO` ou `DESCENDO`.

```
 ──► ( PARADO ) ─abrir─► ( PORTA_ABRINDO ) ─porta_aberta─► ( PORTA_ABERTA ) ─fechar─► ( PORTA_FECHANDO )
        ▲                                                                                │          │
        │                                                                          subir │          │ descer
        │                                                                                ▼          ▼
        │                                                                         ( SUBINDO )  ( DESCENDO )
        │                                                                                │          │
        └──────────────────────────────────── chegar ────────────────────────────────────┴──────────┘
```

Transições extras (laterais):

| De | Evento | Para | Regra de negócio |
|---|---|---|---|
| `PORTA_FECHANDO` | `obstaculo` | `PORTA_ABRINDO` | sensor da porta: alguém no caminho, reabre |
| `PORTA_FECHANDO` | `porta_fechada` | `PARADO` | ninguém chamou, fica parado |
| qualquer estado (menos `EMERGENCIA`) | `emergencia` | `EMERGENCIA` | botão de emergência |
| `EMERGENCIA` | `resolver` | `PARADO` | só a manutenção libera |

Estado de "ciclo terminado" (aceitação): `PARADO`.

In [ ]:
# ⚙️ CÉLULA 7 — Máquina de estados do elevador

delta_elevador = {
    ("PARADO", "abrir"): "PORTA_ABRINDO",
    ("PORTA_ABRINDO", "porta_aberta"): "PORTA_ABERTA",
    ("PORTA_ABERTA", "fechar"): "PORTA_FECHANDO",
    ("PORTA_FECHANDO", "subir"): "SUBINDO",
    ("PORTA_FECHANDO", "descer"): "DESCENDO",
    ("PORTA_FECHANDO", "obstaculo"): "PORTA_ABRINDO",   # sensor da porta: alguém no caminho
    ("PORTA_FECHANDO", "porta_fechada"): "PARADO",      # ninguém chamou: fica parado
    ("SUBINDO", "chegar"): "PARADO",
    ("DESCENDO", "chegar"): "PARADO",
    ("EMERGENCIA", "resolver"): "PARADO",               # só a manutenção libera
}
for _estado in ["PARADO", "PORTA_ABRINDO", "PORTA_ABERTA", "PORTA_FECHANDO", "SUBINDO", "DESCENDO"]:
    delta_elevador[(_estado, "emergencia")] = "EMERGENCIA"

FINAIS_ELEVADOR = {"PARADO"}          # o ciclo terminou: elevador parado de porta fechada

def processar_elevador(eventos, estado_inicial="PARADO", mostrar=True):
    """Roda uma sequência de eventos. Devolve (ok, estado_final)."""
    estado = estado_inicial
    if mostrar:
        print(f"🛗 eventos: {eventos}   (começando em {estado})")
    for i, ev in enumerate(eventos, start=1):
        proximo = delta_elevador.get((estado, ev))
        if proximo is None:
            permitidos = [e for (q, e) in delta_elevador if q == estado]
            if mostrar:
                print(f"   ❌ evento {i} '{ev}' não é permitido em {estado}. Permitidos: {permitidos}\n")
            return False, estado
        if mostrar:
            print(f"   {estado:15} --{ev}--> {proximo}")
        estado = proximo
    ok = estado in FINAIS_ELEVADOR
    if mostrar:
        if ok:
            print(f"   ✅ ciclo finalizado em {estado}\n")
        else:
            print(f"   ⏳ elevador ainda em andamento ({estado})\n")
    return ok, estado

# Reaproveita a classe AFD só para imprimir a tabela e (depois) desenhar o grafo.
elevador = AFD("Máquina de estados do elevador", delta_elevador, "PARADO", FINAIS_ELEVADOR, lambda e: e)
print(f"Estados: {len(elevador.estados)}  |  Transições: {len(delta_elevador)}  |  Eventos: {elevador.alfabeto}\n")

In [ ]:
# ✅ CÉLULA 8 — Dois fluxos VÁLIDOS
processar_elevador(["abrir", "porta_aberta", "fechar", "subir", "chegar"])                                # viagem para cima
processar_elevador(["abrir", "porta_aberta", "fechar", "descer", "emergencia", "resolver"])               # descendo, para na emergência e é liberado

In [ ]:
# ❌ CÉLULA 9 — Dois fluxos INVÁLIDOS (e um extra) — repare que o código lista os eventos permitidos
processar_elevador(["abrir", "subir"])                                                  # tentou subir com a porta abrindo
processar_elevador(["abrir", "porta_aberta", "fechar", "subir", "abrir"])               # tentou abrir a porta com o elevador em movimento
processar_elevador(["emergencia", "abrir"])                                             # em emergência só a manutenção libera

In [ ]:
# 📋 CÉLULA 10 — Eventos permitidos em cada estado (a tabela de transição da máquina)
print("📋 Tabela de transição — Máquina de estados do elevador")
for q in elevador.estados:
    marca = ("→" if q == elevador.inicial else " ") + ("*" if q in elevador.finais else " ")
    saidas = ", ".join(f"{e}→{p}" for (qq, e), p in delta_elevador.items() if qq == q)
    print(f"{marca} {q:15} {saidas}")
print("   (→ = inicial, * = aceitação)")

## 🛰️ Seção 4 — Mini-scanner da linguagem de comandos (R5)

Linguagem **Elevador**. Exemplos: `CHAMAR 7 SUBIR` · `IR S2` · `EMERGENCIA`.

| Token | Padrão | Exemplos |
|---|---|---|
| `RESERVADA` | `CHAMAR IR SUBIR DESCER EMERGENCIA ESPERAR DEFINIR SE MANUTENCAO` | `CHAMAR` |
| `ID` | letra ou `_`, seguida de letras, dígitos ou `_` | `andar`, `velocidade` |
| `NUM_INT` | um ou mais dígitos | `7` |
| `NUM_REAL` | dígitos, ponto, dígitos | `2.5` |
| `ANDAR_TERREO` | a letra `T` sozinha | `T` |
| `ANDAR_SUBSOLO` | `S1` ou `S2` | `S2` |
| `COD_MANUT` | `MNT-` + exatamente 4 dígitos | `MNT-0042` |
| `ATRIB` | `=` | `=` |
| `IGUAL` | `==` | `==` |
| `PONTO_VIRGULA` | `;` | `;` |
| *(ignorados)* | espaços, tabs, quebras de linha e comentários `# ...` | |

**Maximal Munch (3 casos):**

1. `=` × `==`: olhamos um caractere à frente (lookahead).
2. `2` × `2.5`: `NUM_INT` × `NUM_REAL`; o laço só para quando não dá mais para crescer.
3. `MNT` × `MNT-0042`: depois de ler `MNT`, se vem `-` e dígitos, o scanner leva o código **inteiro** como um único token `COD_MANUT`.

**Erros léxicos** informam linha, coluna e dica, e o scanner **continua** depois do erro.
Dicas específicas do domínio: vírgula decimal, hífen fora de código `MNT-` e ponto sem dígitos.

In [ ]:
# 🛰️ CÉLULA 11 — Mini-scanner da linguagem Elevador
# Cada laço "while" é um ESTADO do autômato que fica girando enquanto o caractere pertence ao token.

RESERVADAS = {"CHAMAR", "IR", "SUBIR", "DESCER", "EMERGENCIA", "ESPERAR", "DEFINIR", "SE", "MANUTENCAO"}
DICAS = {
    ",": "use ponto como separador decimal",
    "-": "o hífen só é válido dentro de um código de manutenção, como MNT-0042",
    ".": "número real precisa de dígitos antes e depois do ponto",
}

def eh_letra(c):
    return ("A" <= c <= "Z") or ("a" <= c <= "z") or c == "_"

def eh_digito(c):
    return "0" <= c <= "9"

def escanear(fonte):
    tokens, erros = [], []
    n = len(fonte)
    i, linha, coluna = 0, 1, 1
    while i < n:
        c = fonte[i]
        inicio, col_inicio = i, coluna

        if c == "\n":                                   # quebra de linha: só atualiza a posição
            i, linha, coluna = i + 1, linha + 1, 1
            continue
        if c in " \t\r":                                # espaço: ignorado
            i, coluna = i + 1, coluna + 1
            continue
        if c == "#":                                    # comentário: ignora até o fim da linha
            while i < n and fonte[i] != "\n":
                i += 1
            coluna += i - inicio
            continue

        if eh_letra(c):                                 # estado PALAVRA: letras, dígitos e _
            while i < n and (eh_letra(fonte[i]) or eh_digito(fonte[i])):
                i += 1
            lexema = fonte[inicio:i]
            if lexema == "MNT" and i < n and fonte[i] == "-":      # Maximal Munch: MNT-dddd inteiro
                j = i + 1
                while j < n and eh_digito(fonte[j]):
                    j += 1
                if j - (i + 1) != 4:
                    erros.append(f"Linha {linha}, coluna {col_inicio}: código de manutenção malformado "
                                 f"{fonte[inicio:j]!r} (use MNT- seguido de exatamente 4 dígitos, ex.: MNT-0042)")
                    coluna += j - inicio
                    i = j
                    continue
                i = j
                lexema = fonte[inicio:i]
                tipo = "COD_MANUT"
            elif lexema in RESERVADAS:
                tipo = "RESERVADA"
            elif lexema == "T":
                tipo = "ANDAR_TERREO"
            elif lexema in ("S1", "S2"):
                tipo = "ANDAR_SUBSOLO"
            else:
                tipo = "ID"
        elif eh_digito(c):                              # estado NUM: dígitos e, talvez, .dígitos
            while i < n and eh_digito(fonte[i]):
                i += 1
            tipo = "NUM_INT"
            if i + 1 < n and fonte[i] == "." and eh_digito(fonte[i + 1]):
                i += 1
                while i < n and eh_digito(fonte[i]):
                    i += 1
                tipo = "NUM_REAL"
            lexema = fonte[inicio:i]
        elif c == "=":                                  # "=" ou "==" ? Maximal Munch decide
            if i + 1 < n and fonte[i + 1] == "=":
                i, tipo = i + 2, "IGUAL"
            else:
                i, tipo = i + 1, "ATRIB"
            lexema = fonte[inicio:i]
        elif c == ";":
            i, tipo, lexema = i + 1, "PONTO_VIRGULA", ";"
        else:                                           # nenhum estado aceita: erro léxico
            dica = DICAS.get(c, "caractere fora da linguagem")
            erros.append(f"Linha {linha}, coluna {coluna}: caractere inesperado {c!r} ({dica})")
            i, coluna = i + 1, coluna + 1
            continue

        tokens.append((tipo, lexema, linha, col_inicio))
        coluna += i - inicio
    return tokens, erros


def imprimir_tokens(tokens, erros):
    print(f"{'TIPO':14} {'LEXEMA':10} {'LIN':>3} {'COL':>3}")
    for tipo, lexema, lin, col in tokens:
        print(f"{tipo:14} {lexema:10} {lin:>3} {col:>3}")
    print()
    for e in erros:
        print("⚠️", e)
    if not erros:
        print("✅ nenhum erro léxico")

In [ ]:
# 🧪 CÉLULA 12 — Programa de teste (com erros de propósito nas últimas linhas)
programa = """# roteiro de teste do elevador
DEFINIR velocidade = 1.5
CHAMAR 7 SUBIR
IR S2
IR T
ESPERAR 2.5
SE andar == 7 IR 10;
EMERGENCIA
MANUTENCAO MNT-0042
CHAMAR 3,5 DESCER @
IR MNT-42
IR 5."""

tokens, erros = escanear(programa)
imprimir_tokens(tokens, erros)

tipos = sorted({t[0] for t in tokens})
print(f"\n📊 tipos de token vistos: {len(tipos)} -> {tipos}")
print(f"📊 palavras reservadas vistas: {sorted({t[1] for t in tokens if t[0] == 'RESERVADA'})}")

🔎 **Repare:**

- A linha 1 (comentário) sumiu da tabela: o scanner leu e descartou.
- `3,5` virou `3` e `5`; a vírgula virou erro, com dica para quem digita vírgula decimal.
- `MNT-42` não vira `COD_MANUT`: o scanner leva o lexema todo (`MNT-42`), vê que não são 4 dígitos e reporta o erro.
- `5.` vira `NUM_INT 5` e o ponto sozinho vira erro. O mesmo vale para `.5`.
- O scanner **não parou** no primeiro erro.

In [ ]:
# 🧪 CÉLULA 13 — Os três casos de Maximal Munch, isolados
for texto in ["= ==", "2 2.5", "MNT MNT-0042"]:
    toks, errs = escanear(texto)
    print(f"{texto!r:16} -> {[(t[0], t[1]) for t in toks]}")

## 🧪 Seção 5 — Interface (R6)

Painel com 4 abas: **Validadores**, **Scanner**, **Máquina de estados** e **Roteiro** (a integração da Seção 6).

In [ ]:
# 🧪 CÉLULA 14 — Painel interativo (ipywidgets)
import ipywidgets as widgets
from IPython.display import display

AUTOMATOS = {"Andar": andar, "Código de manutenção": manut}
REGEXES   = {"Andar": REGEX_ANDAR, "Código de manutenção": REGEX_MANUT}

# ---- Aba 1: validadores
v_escolha = widgets.Dropdown(options=list(AUTOMATOS), description="Autômato:")
v_entrada = widgets.Text(value="S2", description="Palavra:")
v_botao   = widgets.Button(description="Validar", button_style="primary")
v_saida   = widgets.Output()

def v_clicar(_):
    v_saida.clear_output()
    with v_saida:
        afd = AUTOMATOS[v_escolha.value]
        afd.rodar(v_entrada.value)
        r_re = re.fullmatch(REGEXES[v_escolha.value], v_entrada.value) is not None
        print(f"RegEx equivalente diz: {'aceita' if r_re else 'rejeita'}")
v_botao.on_click(v_clicar)
aba_validadores = widgets.VBox([widgets.HBox([v_escolha, v_entrada, v_botao]), v_saida])

# ---- Aba 2: scanner
s_entrada = widgets.Textarea(value="CHAMAR 7 SUBIR\nIR S2\nMANUTENCAO MNT-0042\nIR 2,5 @",
                             description="Comandos:", layout=widgets.Layout(width="95%", height="130px"))
s_botao = widgets.Button(description="Escanear", button_style="primary")
s_saida = widgets.Output()

def s_clicar(_):
    s_saida.clear_output()
    with s_saida:
        t, e = escanear(s_entrada.value)
        imprimir_tokens(t, e)
s_botao.on_click(s_clicar)
aba_scanner = widgets.VBox([s_entrada, s_botao, s_saida])

# ---- Aba 3: máquina de estados
m_entrada = widgets.Text(value="abrir, porta_aberta, fechar, subir, chegar",
                         description="Eventos:", layout=widgets.Layout(width="95%"))
m_botao = widgets.Button(description="Processar", button_style="primary")
m_saida = widgets.Output()

def m_clicar(_):
    m_saida.clear_output()
    with m_saida:
        eventos = [e.strip() for e in m_entrada.value.split(",") if e.strip()]
        processar_elevador(eventos)
m_botao.on_click(m_clicar)
aba_maquina = widgets.VBox([m_entrada, m_botao, m_saida])

# ---- Aba 4: roteiro (scanner -> máquina de estados). A função executar_roteiro vem da Seção 6.
r_entrada = widgets.Textarea(value="CHAMAR 7 SUBIR\nIR S2\nIR 25\nEMERGENCIA\nIR T\nMANUTENCAO MNT-0042\nIR T",
                             description="Roteiro:", layout=widgets.Layout(width="95%", height="150px"))
r_botao = widgets.Button(description="Executar", button_style="success")
r_saida = widgets.Output()

def r_clicar(_):
    r_saida.clear_output()
    with r_saida:
        executar_roteiro(r_entrada.value)
r_botao.on_click(r_clicar)
aba_roteiro = widgets.VBox([r_entrada, r_botao, r_saida])

painel = widgets.Tab(children=[aba_validadores, aba_scanner, aba_maquina, aba_roteiro])
for idx, titulo in enumerate(["Validadores", "Scanner", "Máquina", "Roteiro"]):
    painel.set_title(idx, titulo)
display(painel)

## ⭐ Seção 6 — Bônus

### 6.1 AFD mínimo

Se dois estados aceitam exatamente as mesmas continuações, eles são **equivalentes** e podem virar um só.
O algoritmo abaixo (refinamento de partições, da Aula 07) começa separando aceitação × não aceitação e vai
dividindo os grupos até todos os estados de um grupo terem o mesmo comportamento.

No AFD do andar, `AT`, `AD` e `AF` são equivalentes: todos aceitam e nenhum tem seta de saída.

In [ ]:
# ⭐ CÉLULA 15 — Minimização de AFD (refinamento de partições)

def minimizar(afd):
    MORTO = "__morto__"
    Q = set(afd.estados) | {MORTO}
    sigma = afd.alfabeto

    def d(q, s):
        return MORTO if q == MORTO else afd.delta.get((q, s), MORTO)

    grupo = {q: (q in afd.finais) for q in Q}                 # partição inicial: F × (Q − F)
    while True:
        assinatura = {q: (grupo[q], tuple(grupo[d(q, s)] for s in sigma)) for q in Q}
        ids = {}
        novo = {q: ids.setdefault(assinatura[q], len(ids)) for q in Q}
        if len(set(novo.values())) == len(set(grupo.values())):
            grupo = novo
            break
        grupo = novo

    nome = {}
    for g in set(grupo.values()):
        membros = sorted(q for q in Q if grupo[q] == g and q != MORTO)
        nome[g] = "|".join(membros) if membros else None     # grupo só com o estado morto
    novo_delta = {}
    for (q, s), p in afd.delta.items():
        if nome[grupo[q]] and nome[grupo[p]]:
            novo_delta[(nome[grupo[q]], s)] = nome[grupo[p]]
    finais = {nome[grupo[q]] for q in afd.finais}
    return AFD(afd.nome + " (mínimo)", novo_delta, nome[grupo[afd.inicial]], finais, afd.classificar)


andar_min = minimizar(andar)
print(f"Andar:      {len(andar.estados)} estados  ->  {len(andar_min.estados)} estados no AFD mínimo\n")
andar_min.tabela()

# O mínimo tem que continuar equivalente à RegEx:
div_min = bateria(andar_min, REGEX_ANDAR, ACEITAS_ANDAR, REJEITADAS_ANDAR)

manut_min = minimizar(manut)
print(f"Manutenção: {len(manut.estados)} estados  ->  {len(manut_min.estados)} estados (já era mínimo: nada para juntar)")

### 6.2 Desenho automático com Graphviz

O desenho é gerado a partir do próprio `delta`. Setas com a mesma origem e destino são juntadas em uma só (`0,1,2`).

In [ ]:
# ⭐ CÉLULA 16 — Desenho automático do AFD com Graphviz (a partir do delta)
from IPython.display import SVG

def para_dot(afd):
    juntas = {}
    for (q, s), p in afd.delta.items():
        juntas.setdefault((q, p), []).append(s)
    linhas = ["digraph G {", "  rankdir=LR;", '  node [shape=circle, fontsize=11];',
              '  __ini [shape=point];', f'  __ini -> "{afd.inicial}";']
    for q in afd.estados:
        if q in afd.finais:
            linhas.append(f'  "{q}" [shape=doublecircle];')
    for (q, p), simbolos in juntas.items():
        linhas.append(f'  "{q}" -> "{p}" [label="{", ".join(simbolos)}"];')
    linhas.append("}")
    return "\n".join(linhas)

def desenhar(afd):
    dot = para_dot(afd)
    try:
        import graphviz
        svg = graphviz.Source(dot).pipe(format="svg").decode("utf-8")
        print(f"🎨 {afd.nome}")
        display(SVG(svg))
    except Exception as erro:                       # sem Graphviz: mostra o código DOT mesmo assim
        print(f"(Graphviz indisponível: {erro}). Código DOT gerado:\n")
        print(dot)

desenhar(andar)
desenhar(andar_min)
desenhar(manut)
desenhar(elevador)

### 6.3 Tudo conectado: o scanner lê um roteiro e os tokens viram eventos da máquina de estados

O roteiro é escaneado; cada comando vira uma sequência de **eventos** do elevador:

| Comando | O que acontece |
|---|---|
| `IR x` / `CHAMAR x SUBIR\|DESCER` | valida `x` com o **AFD do andar**; gera `abrir, porta_aberta, fechar, subir/descer, chegar` |
| `EMERGENCIA` | gera o evento `emergencia` |
| `MANUTENCAO MNT-dddd` | valida o código com o **AFD de manutenção**; gera `resolver` |
| `DEFINIR`, `ESPERAR`, `SE` | só passam pelo scanner (não executam nada) |

Numeração interna dos andares: `T` = 0, `S1` = −1, `S2` = −2, números = ele mesmo.

In [ ]:
# ⭐ CÉLULA 17 — Tudo conectado: scanner -> tokens -> eventos -> máquina de estados

TIPOS_ANDAR = {"NUM_INT", "ANDAR_TERREO", "ANDAR_SUBSOLO"}

def numero_do_andar(lexema):
    return {"T": 0, "S1": -1, "S2": -2}.get(lexema) if lexema in ("T", "S1", "S2") else int(lexema)

def eventos_da_viagem(origem, destino):
    if origem == destino:
        return ["abrir", "porta_aberta", "fechar", "porta_fechada"]
    sentido = "subir" if destino > origem else "descer"
    return ["abrir", "porta_aberta", "fechar", sentido, "chegar"]

def executar_roteiro(fonte):
    tokens, erros = escanear(fonte)
    for e in erros:
        print("⚠️ léxico:", e)
    estado, andar_atual = "PARADO", 0
    print(f"🏢 início: estado={estado}, andar=T (0)\n")
    i = 0
    while i < len(tokens):
        tipo, lex, lin, col = tokens[i]
        proximo = tokens[i + 1] if i + 1 < len(tokens) else None

        if tipo == "RESERVADA" and lex in ("IR", "CHAMAR"):
            if proximo is None or proximo[0] not in TIPOS_ANDAR:
                print(f"❌ linha {lin}: {lex} precisa de um andar logo depois\n")
                i += 1
                continue
            alvo = proximo[1]
            i += 2
            if i < len(tokens) and tokens[i][1] in ("SUBIR", "DESCER") and lex == "CHAMAR":
                i += 1                                          # direção da chamada (informativa)
            if not andar.rodar(alvo, mostrar=False):            # AFD do andar decide
                print(f"❌ linha {lin}: {lex} {alvo} -> andar inválido segundo o AFD (válidos: T, S1, S2, 1 a 20)\n")
                continue
            destino = numero_do_andar(alvo)
            print(f"▶ linha {lin}: {lex} {alvo}  (andar {andar_atual} -> {destino})")
            ok, estado = processar_elevador(eventos_da_viagem(andar_atual, destino), estado)
            if ok:
                andar_atual = destino
        elif tipo == "RESERVADA" and lex == "EMERGENCIA":
            print(f"▶ linha {lin}: EMERGENCIA")
            _, estado = processar_elevador(["emergencia"], estado)
            i += 1
        elif tipo == "RESERVADA" and lex == "MANUTENCAO":
            if proximo is None or proximo[0] != "COD_MANUT" or not manut.rodar(proximo[1], mostrar=False):
                print(f"❌ linha {lin}: MANUTENCAO precisa de um código válido (MNT-0000)\n")
                i += 1
                continue
            print(f"▶ linha {lin}: MANUTENCAO {proximo[1]}  (código validado pelo AFD)")
            _, estado = processar_elevador(["resolver"], estado)
            i += 2
        else:
            if tipo == "RESERVADA":
                print(f"ℹ️ linha {lin}: '{lex}' passou pelo scanner, mas não executa nada\n")
            i += 1
    print(f"🏁 fim: estado={estado}, andar={andar_atual}")

roteiro = """CHAMAR 7 SUBIR
IR S2
IR 25
EMERGENCIA
IR T
MANUTENCAO MNT-0042
IR T"""
executar_roteiro(roteiro)

## ✅ Resumo final e checklist

In [ ]:
# ✅ CÉLULA 18 — Resumo dos testes
total = div_andar + div_manut + div_min
print(f"Divergências AFD × RegEx  (andar): {div_andar}")
print(f"Divergências AFD × RegEx  (manutenção): {div_manut}")
print(f"Divergências AFD mínimo × RegEx (andar): {div_min}")
print(f"Palavra vazia testada: {'' in REJEITADAS_ANDAR and '' in REJEITADAS_MANUT}")
assert total == 0, "Há divergências entre AFD e RegEx!"
print("\n🎉 Tudo certo: 0 divergências em todos os autômatos.")

| Item | Onde |
|---|---|
| R1 — 2 AFDs com ≥ 5 estados, desenho, `tabela()` | Seções 1 e 2 (7 e 9 estados) |
| R2 — rastreamento (1 aceita + 1 rejeitada, com explicação) | Células 3, 5 e tabela "Por que foram rejeitadas?" |
| R3 — bateria AFD × RegEx, ≥ 5 + 5, vazia, 0 divergências | Células 4 e 6 |
| R4 — máquina com ≥ 5 estados e ≥ 6 transições, 2 fluxos válidos e 2 inválidos | Seção 3 (7 estados, 16 transições) |
| R5 — scanner com ≥ 8 tokens, ≥ 3 reservadas, Maximal Munch, erros com linha/coluna/dica | Seção 4 (10 tipos, 9 reservadas, 3 casos de Maximal Munch) |
| R6 — interface | Seção 5 |
| R7 — README | `README.md` do repositório |
| Bônus — AFD mínimo, Graphviz, tudo conectado | Seção 6 |